# LiDAR 정면 거리로 장애물 앞에서 정지

LiDAR 정면 거리 측정과 모터 제어(PWM)를 합쳐, 앞으로 전진하다가 장애물이 가까워지면 멈추는 동작을 구현했습니다.

## 하는 일
- 정면 0° ±10°의 거리 중앙값을 0.1초마다 갱신하고, 300mm 이하가 되면 모터를 정지합니다.
- LiDAR 읽기는 별도 스레드, 모터 제어는 메인 루프에서 합니다.

## 안전장치
- 출발 전에 정면 거리를 읽지 못하면 출발하지 않습니다.
- 주행 중 정면 값이 0.5초 이상 갱신되지 않거나 LiDAR 스레드에 오류가 나면 정지합니다.
- 최대 주행 시간(`MAX_RUN_SECONDS`, 기본 20초)이 지나면 정지하고, 중단(■)이나 오류 때도 `finally`에서 정지합니다.
- `SIMULATE` 설정(`"error"` 또는 `"stale"`)으로 LiDAR 오류와 값 갱신 중단을 가짜로 일으켜 안전장치를 시험할 수 있습니다. 평소에는 `None`입니다.

## 확인한 것
- 바퀴를 띄운 상태에서 앞을 손으로 막으면 정지했고, 바닥에서는 장애물 앞에서 정지했습니다.
- 정상 실행 한 번: 출발 시 정면 780mm, 294mm에서 감지, 정지 후 275mm. 기준(300mm)보다 약 25mm 더 진행했습니다.
- 접근 속도는 출력 간격으로 계산한 추정치로 약 8~9cm/s입니다.
- `SIMULATE = "error"`: 출발 약 3초 후 `LiDAR 오류로 정지`가 출력되며 멈췄습니다.
- `SIMULATE = "stale"`: 값 갱신이 멈추자 `정면 거리를 읽지 못해 정지`가 출력되며 멈췄습니다.
- `MAX_RUN_SECONDS = 3`: 796mm에서 출발해 3초 뒤 `최대 주행 시간 도달 -> 정지`로 멈췄고, 정지 후 521mm였습니다.
- 거리는 LiDAR 중심 기준이며, 로봇 앞면과 장애물 사이의 실제 간격은 재지 않았습니다.

## 아직 확인하지 않은 것
- `DRIVE_POWER`를 바꿨을 때의 정지 거리 (관성이 달라짐)
- 중앙값 방식이 의자 다리처럼 얇은 물체를 놓치는지
- 무선(Wi-Fi) 연결이 끊긴 상태에서의 동작

In [6]:
import time
import threading
import statistics
from rplidar import RPLidar, RPLidarException
from tiki.mini import TikiMini

# ---- 설정 ----
PORT = '/dev/ttyUSB0'
BAUDRATE = 460800
STOP_DISTANCE_MM = 300     # 정면 거리가 이 값 이하이면 정지
DRIVE_POWER = 80           # PWM (0~255). 낮게 시작
FRONT_WINDOW_DEG = 10      # 정면 ± 도
UPDATE_INTERVAL = 0.1      # 정면 거리 갱신 주기(초)
STALE_LIMIT = 0.5          # 이 시간 동안 새 값이 없으면 정지 (안전장치)
MAX_RUN_SECONDS = 20       # 최대 주행 시간 (안전장치)

# ---- 안전장치 시험용 (평소에는 None) ----
SIMULATE = None            # None | "error" | "stale"
SIMULATE_AFTER = 3.0       # 첫 LiDAR 값을 받은 뒤 이 시간(초)이 지나면 가짜 고장 발생

front = {"mm": None, "t": 0.0}
status = {"msg": ""}
error = None
stop_event = threading.Event()


def measures_with_retry(lidar, retries=5):
    for attempt in range(1, retries + 1):
        gen = lidar.iter_measures()
        try:
            first = next(gen)
        except RPLidarException as e:
            status["msg"] = f"스캔 시작 실패 ({attempt}/{retries}): {e}"
            try:
                lidar.stop()
            except Exception:
                pass
            time.sleep(1.0)
            continue
        status["msg"] = f"스캔 중 (시작 시도 {attempt}회)"
        yield first
        yield from gen
        return
    raise RuntimeError("LiDAR 스캔 시작에 계속 실패했습니다")


def in_front(angle):
    diff = abs((angle + 180) % 360 - 180)      # 0도와의 각도 차이
    return diff <= FRONT_WINDOW_DEG


def lidar_worker(lidar):
    global error
    buf = []
    last_update = time.time()
    worker_start = None
    try:
        for _, quality, angle, distance in measures_with_retry(lidar):
            if worker_start is None:
                worker_start = time.time()

            # 안전장치 시험용: 가짜 고장 (SIMULATE가 None이면 아무 일도 안 함)
            if SIMULATE and time.time() - worker_start > SIMULATE_AFTER:
                if SIMULATE == "error":
                    raise RuntimeError("시뮬레이션: LiDAR 오류")
                if SIMULATE == "stale":        # 오류 없이 값 갱신만 멈춤
                    while not stop_event.is_set():
                        time.sleep(0.1)
                    break

            if stop_event.is_set():
                break
            if distance > 0 and in_front(angle):
                buf.append(distance)
            now = time.time()
            if now - last_update > UPDATE_INTERVAL:
                if buf:
                    front["mm"] = statistics.median(buf)
                    front["t"] = now
                else:
                    front["mm"] = None          # 유효값 없음 = 알 수 없음
                buf.clear()
                last_update = now
    except Exception as e:
        error = e


robot = TikiMini()
robot.set_motor_mode(robot.MOTOR_MODE_PWM)
lidar = RPLidar(PORT, baudrate=BAUDRATE, timeout=2)
worker = threading.Thread(target=lidar_worker, args=(lidar,), daemon=True)
worker.start()

try:
    # 정면 거리를 처음 읽을 때까지 대기 (모터는 아직 정지 상태)
    t0 = time.time()
    while front["mm"] is None and time.time() - t0 < 10:
        time.sleep(0.1)
    if front["mm"] is None:
        raise RuntimeError(f"정면 거리를 읽지 못해 주행하지 않습니다 ({status['msg']})")

    print(f"출발: 정면 {front['mm']:.0f} mm, {STOP_DISTANCE_MM} mm 이하면 정지")
    t_start = last_print = time.time()
    robot.forward(DRIVE_POWER)

    while True:
        time.sleep(0.05)
        now = time.time()
        mm = front["mm"]

        if error is not None:
            print("LiDAR 오류로 정지:", error)
            break
        if mm is None or now - front["t"] > STALE_LIMIT:
            print("정면 거리를 읽지 못해 정지")
            break
        if mm <= STOP_DISTANCE_MM:
            print(f"장애물 감지: {mm:.0f} mm -> 정지")
            break
        if now - t_start > MAX_RUN_SECONDS:
            print("최대 주행 시간 도달 -> 정지")
            break
        if now - last_print > 0.3:
            print(f"정면 {mm:.0f} mm")
            last_print = now

    robot.stop()
    time.sleep(0.7)                              # 관성으로 더 움직인 뒤의 거리
    if front["mm"] is not None:
        print(f"정지 후 정면 거리: {front['mm']:.0f} mm")

except KeyboardInterrupt:
    print("\n사용자 중단")

finally:
    try:
        robot.stop()
    except Exception:
        pass
    stop_event.set()
    worker.join(timeout=3)
    for fn in (lidar.stop, lidar.stop_motor, lidar.disconnect):
        try:
            fn()
        except Exception:
            pass
    print("종료")

출발: 정면 803 mm, 300 mm 이하면 정지
정면 794 mm
정면 774 mm
정면 750 mm
정면 716 mm
정면 692 mm
정면 663 mm
정면 635 mm
정면 602 mm
정면 574 mm
정면 548 mm
정면 520 mm
정면 492 mm
정면 461 mm
정면 429 mm
정면 398 mm
정면 371 mm
정면 342 mm
정면 314 mm
장애물 감지: 294 mm -> 정지
정지 후 정면 거리: 280 mm
종료
